In [2]:
import os 
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms

1-Get Device for Training

We want to be able to train our model on an accelerator such as CUDA, MPS, MTIA, or XPU. If the current accelerator is available, we will use it. Otherwise, we use the CPU.

In [7]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


Define the Class (owr neural network)
We define our neural network by subclassing nn.Module, and initialize the neural network layers in __init__. Every nn.Module subclass implements the operations on input data in the forward method.

thanks to the nn.module

It allows PyTorch to:
save layers
manage settings
save the model
send the model to the GPU
so we ne to inherite from 

In [8]:
from torch import nn

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()

        self.flatten = nn.Flatten()

        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU()
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

Flatten Layer :we have images [1,28,28]
a linear layer expectes a 1D vector 
so flatten converts 1*28*28 vector [ 784 ]

In [ ]:
#self.linear_relu_stack = nn.Sequential(...)

Sequential Block
nn.sequential allows us to chain layers togethers
nn.linear(784,512)
create a fully connected layer input 784 output 512

ReLU Activation
It introduces non-linearity, allowing the network to learn complex patterns.

Second Linear Layer
#nn.Linear(512, 512)
input 512 neurons
output 512 neurons


Second RELU

Forward Method
The forward() method defines how data moves through the network.
whenever you do output = model(images)
pytorch automatically calls forward(image)

Pass Through Layers

In [ ]:
self.linear_relu_stack = nn.Sequential(
    nn.Linear(28 * 28, 512),
    nn.ReLU(),
    nn.Linear(512, 512),
    nn.ReLU()
)

nn.sequentail stores several layers and executes them one after another.
so when you write 
logits = self.linear_relu_stack(x)
PyTorch automatically does:
    x = Linear1(x)
    x = ReLU(x)
    x = Linear2(x)
    x = ReLU(x)

Logits are the raw scores produced by the network before converting them into probabilities.

exemple tensor([
    2.5,
   -1.2,
    0.8,
    4.1
])
the larget value is 4.1 which par exemple mean class horse 
we use sofmax to convert to probabiltes 
probs = torch.softmax(logits, dim=0)


In training, PyTorch loss functions like nn.CrossEntropyLoss() usually expect logits directly, not probabilities. That's why the model returns logits instead of applying Softmax inside forward().

--------------------------------------------------------------------------

we create a instance of Neural  Network and move it to the device and print its stucutre 

In [9]:
model = NeuralNetwork().to(device)

To use the model, we pass it the input data. This executes the model’s forward, along with some background operations. Do not call model.forward() directly!

Calling the model on the input returns a 2-dimensional tensor with dim=0 corresponding to each output of 10 raw predicted values for each class, and dim=1 corresponding to the individual values of each output. We get the prediction probabilities by passing it through an instance of the nn.Softmax module.

In [15]:
X = torch.rand(1, 28 ,28,device=device)#random tensor with shape (1,28,28)
logits = model(X)
pred_prob = nn.Softmax(dim=1)(logits)
y_pred = pred_prob.argmax(1)
print(f"Predicted class: {y_pred}")


Predicted class: tensor([222], device='cuda:0')


Model Layers

Let’s break down the layers in the FashionMNIST model. To illustrate it, we will take a sample minibatch of 3 images of size 28x28 and see what happens to it as we pass it through the network.

In [22]:
input_image = torch.rand(3,28, 28)
print(f"Image shape: {input_image.shape}")

Image shape: torch.Size([3, 28, 28])


nn.Flatten
We initialize the nn.Flatten layer to convert each 2D 28x28 image into a contiguous array of 784 pixel values ( the minibatch dimension (at dim=0) is maintained).

In [23]:
flatten = nn.Flatten()
flat_image = flatten(input_image)
print(f"Flattened image shape: {flat_image.shape}")

Flattened image shape: torch.Size([3, 784])


nn.Linear is a fully connected (dense) layer in a neural network.

It applies a linear transformation to the input using (weights and biases) that are learned during training.
A linear layer computes:

y = x*transpose(W)+b
that is why 
    nn.linear(len(x),len(y))

where:

x = input
W = weight matrix (learned)
b = bias vector (learned)
y = output

In [24]:
layer1 = nn.Linear(in_features=28*28, out_features=20)
output1 = layer1(flat_image)
print(f"output1: {output1}")

output1: tensor([[ 0.1172, -0.2464,  0.3248,  0.1534, -0.1855,  0.0945,  0.0869, -0.3409,
          0.1219, -0.1046,  0.2132,  0.0432,  0.0152, -0.2610,  0.1185,  0.1928,
         -0.3959, -0.1866,  0.1247, -0.5486],
        [ 0.3601, -0.3113,  0.4318,  0.3748, -0.4698,  0.2192,  0.0084,  0.2976,
          0.3566, -0.0792,  0.4789,  0.4099, -0.1378, -0.2930,  0.1393,  0.0532,
         -0.2524, -0.5276, -0.0701, -0.6045],
        [-0.0088,  0.0685,  0.1512,  0.1437, -0.6561, -0.0133,  0.1985,  0.0688,
          0.2056,  0.2737,  0.1765,  0.3868,  0.1831, -0.3833, -0.2559, -0.1622,
          0.0060,  0.0510,  0.1661, -0.5165]], grad_fn=<AddmmBackward0>)


nn.ReLU

Non-linear activations are what create the complex mappings between the model’s inputs and outputs. They are applied after linear transformations to introduce nonlinearity, helping neural networks learn a wide variety of phenomena.

In this model, we use nn.ReLU between our linear layers, but there’s other activations to introduce non-linearity in your model.

In [25]:
print(f'before ReLU: {output1}')
relu = nn.ReLU()
output2 = relu(output1)
print(f'after ReLU: {output2}')


before ReLU: tensor([[ 0.1172, -0.2464,  0.3248,  0.1534, -0.1855,  0.0945,  0.0869, -0.3409,
          0.1219, -0.1046,  0.2132,  0.0432,  0.0152, -0.2610,  0.1185,  0.1928,
         -0.3959, -0.1866,  0.1247, -0.5486],
        [ 0.3601, -0.3113,  0.4318,  0.3748, -0.4698,  0.2192,  0.0084,  0.2976,
          0.3566, -0.0792,  0.4789,  0.4099, -0.1378, -0.2930,  0.1393,  0.0532,
         -0.2524, -0.5276, -0.0701, -0.6045],
        [-0.0088,  0.0685,  0.1512,  0.1437, -0.6561, -0.0133,  0.1985,  0.0688,
          0.2056,  0.2737,  0.1765,  0.3868,  0.1831, -0.3833, -0.2559, -0.1622,
          0.0060,  0.0510,  0.1661, -0.5165]], grad_fn=<AddmmBackward0>)
after ReLU: tensor([[0.1172, 0.0000, 0.3248, 0.1534, 0.0000, 0.0945, 0.0869, 0.0000, 0.1219,
         0.0000, 0.2132, 0.0432, 0.0152, 0.0000, 0.1185, 0.1928, 0.0000, 0.0000,
         0.1247, 0.0000],
        [0.3601, 0.0000, 0.4318, 0.3748, 0.0000, 0.2192, 0.0084, 0.2976, 0.3566,
         0.0000, 0.4789, 0.4099, 0.0000, 0.0000, 0.1393

nn.Sequential

nn.Sequential is a simple way to build a neural network by stacking layers in order.
It is an ordered container of modules, meaning:
👉 data flows through each layer step by step, in the same order you define them

Inside nn.Sequential:

✔ Output of one layer becomes input of the next
❌ You cannot skip layers
❌ You cannot branch (for complex networks)

Not good for:
Complex architectures (ResNet, Transformers)
Multiple inputs/outputs
Skip connections

For those, you use a custom forward() function.

In [26]:
seq_modules = nn.Sequential(
    flatten,
    layer1,
    nn.ReLU(),
    nn.Linear(20, 10)
)
input_image = torch.rand(3,28,28)
logits = seq_modules(input_image)

SOFTMAX
The last linear layer of the neural network returns logits - raw values in [-infty, infty] - which are passed to the nn.Softmax module. The logits are scaled to values [0, 1] representing the model’s predicted probabilities for each class. dim parameter indicates the dimension along which the values must sum to 1.

In [27]:
softmax = nn.Softmax(dim=1)
pred_prob = softmax(logits)
y_pred = pred_prob.argmax(1)
print(f"Predicted class: {y_pred}")

Predicted class: tensor([2, 2, 2])


parameterized layers
In a neural network, some layers have learnable values:

weights
biases
These are called parameters.

PyTorch automatically:
✔ detects layers
✔ registers their parameters
✔ stores them inside the model

What does “tracks all fields” mean?
PyTorch sees everything defined inside the class and organizes it.
But only layers with parameters are tracked as trainable.

| Layer   | Has parameters? |
| ------- | --------------- |
| Linear  | ✅ Yes          |
| Conv2d  | ✅ Yes          |
| ReLU    | ❌ No           |
| Flatten | ❌ No           |


model.parameters()
This method gives you all trainable parameters in the model.

named_parameters()
This is similar but gives names + values.
These are used by optimizers to train the model

In [28]:
print(f"Model structure: {model}")

for name , parm in model.named_parameters():
    print(f"Layer: {name} | Size: {parm.size()} | Values : {parm[:2]} \n")

Model structure: NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
  )
)
Layer: linear_relu_stack.0.weight | Size: torch.Size([512, 784]) | Values : tensor([[ 0.0350, -0.0235,  0.0351,  ...,  0.0100, -0.0041, -0.0317],
        [ 0.0044,  0.0101, -0.0101,  ..., -0.0254,  0.0294,  0.0005]],
       device='cuda:0', grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.0.bias | Size: torch.Size([512]) | Values : tensor([-0.0139, -0.0029], device='cuda:0', grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.weight | Size: torch.Size([512, 512]) | Values : tensor([[-0.0305,  0.0001, -0.0156,  ...,  0.0161, -0.0302,  0.0350],
        [ 0.0224, -0.0212, -0.0141,  ...,  0.0438, -0.0257, -0.0221]],
       device='cuda:0', grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.bias | Size: torch.

This prints the architecture of your neural network.

What you learn from it:
What layers exist
The order of layers
Input/output sizes
Whether layers have bias

👉 This is useful for debugging and understanding your model.

Looping over parameters:
model.named_parameters() returns:
    name → the layer parameter name
    parm → the actual tensor (weights or biases)

exemples
linear_relu_stack.0.weight
linear_relu_stack.0.bias
linear_relu_stack.2.weight
linear_relu_stack.2.bias

for each parametres print parameter info 
    name
    parm.size it can be weight or bias 


for more https://docs.pytorch.org/docs/2.12/nn.html
